# MovieRec3 — Full Training Pipeline (Kaggle GPU)

Notebook này chạy toàn bộ pipeline huấn luyện:
1. Train PDF-clean artifacts (SBERT + LightGCN + Two-Tower) cho MovieLens + Letterboxd
2. Train Strong Ranker cho cả hai dataset
3. Comparison suite (core + full)
4. Embedding visualization
5. Audit & zip outputs

**Yêu cầu:** Bật GPU accelerator trong Settings. Upload `movierec3_kaggle_input.zip` làm dataset.

In [ ]:
import os, sys, shutil, subprocess, json, zipfile
from pathlib import Path

kaggle_input = Path('/kaggle/input')
project_dir = Path('/kaggle/working/movierec3')

if kaggle_input.exists():
    datasets = [d for d in kaggle_input.iterdir() if d.is_dir()]
    if datasets and not project_dir.exists():
        input_dir = datasets[0]
        print(f'Copying from {input_dir} to {project_dir}')
        shutil.copytree(str(input_dir), str(project_dir))

os.chdir(project_dir)
os.environ['PYTHONPATH'] = f'{project_dir}/src:{project_dir}'
os.environ['PYTHONUNBUFFERED'] = '1'
print('Working dir:', os.getcwd())
!ls data/raw/ data/processed/

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-optional.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    os.environ['DEVICE'] = 'cuda'
    os.environ['EPOCHS'] = '100'
    os.environ['BATCH'] = '8192'
else:
    print('WARNING: No GPU')
    os.environ['DEVICE'] = 'cpu'
    os.environ['EPOCHS'] = '20'
    os.environ['BATCH'] = '4096'

os.environ['DIM'] = '128'
os.environ['GRID'] = '0.05'
os.environ['SBERT'] = 'sentence-transformers/all-mpnet-base-v2'
print(f"Config: device={os.environ['DEVICE']}, epochs={os.environ['EPOCHS']}, batch={os.environ['BATCH']}")

## Phase 1: Train PDF-Clean Artifacts

In [ ]:
# MovieLens PDF-clean
!PYTHONPATH=src:. python -u scripts/train.py \
  --raw-dir data/raw/ml-latest-small \
  --enriched-catalog data/processed/movie_catalog_enriched.parquet \
  --artifacts-dir artifacts/movielens_pdf_clean \
  --content-backend sbert \
  --sbert-model $SBERT \
  --train-lightgcn --train-two-tower \
  --lightgcn-dim $DIM --lightgcn-layers 3 \
  --epochs $EPOCHS --batch-size $BATCH \
  --device $DEVICE --hybrid-grid-step $GRID \
  --min-rating 4.0

In [ ]:
# Letterboxd PDF-clean
!PYTHONPATH=src:. python -u scripts/train.py \
  --raw-dir data/processed/letterboxd \
  --enriched-catalog data/processed/letterboxd/movie_catalog_enriched.parquet \
  --artifacts-dir artifacts/letterboxd_pdf_clean \
  --content-backend sbert \
  --sbert-model $SBERT \
  --train-lightgcn --train-two-tower \
  --lightgcn-dim $DIM --lightgcn-layers 3 \
  --epochs $EPOCHS --batch-size $BATCH \
  --device $DEVICE --hybrid-grid-step $GRID \
  --min-rating 4.0

## Phase 2: Train Strong Ranker

In [ ]:
# MovieLens Strong
!PYTHONPATH=src:. python -u scripts/train_strong_hybrid.py \
  --dataset movielens \
  --raw-dir data/raw/ml-latest-small \
  --enriched-catalog data/processed/movie_catalog_enriched.parquet \
  --artifacts-dir artifacts/movielens_strong \
  --content-backend sbert --sbert-model $SBERT \
  --ranker lightgbm \
  --lightgcn-dim $DIM --lightgcn-layers 3 \
  --lightgcn-epochs $EPOCHS --batch-size $BATCH \
  --device $DEVICE \
  --max-ease-items 5000 --max-ranker-samples 500000 \
  --min-rating 4.0

In [ ]:
# Letterboxd Strong
!PYTHONPATH=src:. python -u scripts/train_strong_hybrid.py \
  --dataset letterboxd \
  --raw-dir data/processed/letterboxd \
  --enriched-catalog data/processed/letterboxd/movie_catalog_enriched.parquet \
  --artifacts-dir artifacts/letterboxd_strong \
  --content-backend sbert --sbert-model $SBERT \
  --ranker lightgbm \
  --lightgcn-dim $DIM --lightgcn-layers 3 \
  --lightgcn-epochs $EPOCHS --batch-size $BATCH \
  --device $DEVICE \
  --max-ease-items 5000 --max-ranker-samples 500000 \
  --min-rating 4.0

## Phase 3: Comparison Suite

In [ ]:
# Core comparison (both datasets)
!PYTHONPATH=src:. python -u scripts/compare_models.py \
  --dataset both \
  --movielens-dir data/raw/ml-latest-small \
  --movielens-enriched-catalog data/processed/movie_catalog_enriched.parquet \
  --letterboxd-dir data/processed/letterboxd \
  --letterboxd-enriched-catalog data/processed/letterboxd/movie_catalog_enriched.parquet \
  --content-backend sbert --sbert-model $SBERT \
  --preset letterboxd-pdf-clean \
  --models core \
  --k 10 --epochs $EPOCHS --mf-dim $DIM --batch-size $BATCH \
  --device $DEVICE --hybrid-grid-step $GRID \
  --output-dir reports/comparison_sbert_pdf_clean_both

In [ ]:
# Full comparison (both datasets)
!PYTHONPATH=src:. python -u scripts/compare_models.py \
  --dataset both \
  --movielens-dir data/raw/ml-latest-small \
  --movielens-enriched-catalog data/processed/movie_catalog_enriched.parquet \
  --letterboxd-dir data/processed/letterboxd \
  --letterboxd-enriched-catalog data/processed/letterboxd/movie_catalog_enriched.parquet \
  --content-backend sbert --sbert-model $SBERT \
  --preset letterboxd-strong \
  --models full \
  --k 10 --epochs $EPOCHS --mf-dim $DIM --batch-size $BATCH \
  --device $DEVICE \
  --max-ease-items 5000 --max-slim-items 3000 --max-ranker-samples 500000 \
  --hybrid-grid-step $GRID \
  --output-dir reports/comparison_sbert_strong_both

## Phase 4: Visualization & Audit

In [ ]:
!PYTHONPATH=src:. python -u scripts/visualize_embeddings.py \
  --artifacts-dir artifacts/movielens_pdf_clean \
  --output-dir reports/embedding_visualization_movielens \
  --method tsne --sample-size 2500 --top-genres 8

!PYTHONPATH=src:. python -u scripts/visualize_embeddings.py \
  --artifacts-dir artifacts/letterboxd_pdf_clean \
  --output-dir reports/embedding_visualization_letterboxd \
  --method tsne --sample-size 2500 --top-genres 8

In [ ]:
!PYTHONPATH=src:. python -u scripts/audit_artifacts.py

In [ ]:
for md_path in [
    'reports/comparison_sbert_pdf_clean_both/comparison_summary.md',
    'reports/comparison_sbert_strong_both/comparison_summary.md',
]:
    p = Path(md_path)
    if p.exists():
        print('\n' + '='*60)
        print(md_path)
        print('='*60)
        print(p.read_text())

for m in [
    'artifacts/movielens_pdf_clean/metrics.json',
    'artifacts/letterboxd_pdf_clean/metrics.json',
]:
    p = Path(m)
    if p.exists():
        data = json.loads(p.read_text())
        print(f'\n--- {m} ---')
        for split in ['validation', 'test']:
            if split in data:
                print(f'  {split}:')
                for k, v in data[split].items():
                    print(f'    {k}: {v:.4f}')

## Phase 5: Zip & Download

In [ ]:
zip_paths = [
    'artifacts/movielens_pdf_clean',
    'artifacts/letterboxd_pdf_clean',
    'artifacts/movielens_strong',
    'artifacts/letterboxd_strong',
    'reports/comparison_sbert_pdf_clean_both',
    'reports/comparison_sbert_strong_both',
    'reports/embedding_visualization_movielens',
    'reports/embedding_visualization_letterboxd',
]

output_zip = '/kaggle/working/full_training_outputs.zip'
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for base_path in zip_paths:
        p = Path(base_path)
        if p.exists():
            for f in p.rglob('*'):
                if f.is_file():
                    zf.write(f, str(f))
            print(f'  Zipped: {base_path}')
        else:
            print(f'  Skipped: {base_path}')

print(f'\nOutput: {output_zip}')
print(f'Size: {os.path.getsize(output_zip) / 1024 / 1024:.1f} MB')

from IPython.display import FileLink
FileLink(output_zip)